In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Optional, Any
import mlflow
import mlflow.pyfunc
import re
import math
from bs4 import BeautifulSoup

# --------------------------------------------------
# ✅ INIT API
# --------------------------------------------------
app = FastAPI(title="Rakuten Product Classification API")

# --------------------------------------------------
# ✅ MLflow CONFIG
# --------------------------------------------------
mlflow.set_tracking_uri("file:///C:/Users/user/Rakuten-Challenge/mlruns")

# ✅ modèle en production
model = mlflow.pyfunc.load_model("models:/rakuten_model@prod")

# --------------------------------------------------
# ✅ DATA MODEL
# --------------------------------------------------
class Product(BaseModel):
    designation: str
    description: Optional[str] = ""

# --------------------------------------------------
# ✅ CLEANING HELPERS (robuste)
# --------------------------------------------------
def safe_to_str(x: Any) -> str:
    """
    Convertit n'importe quel type en string propre
    et gère NaN/None.
    """
    if x is None:
        return ""
    # numpy/pandas NaN -> float('nan')
    if isinstance(x, float) and math.isnan(x):
        return ""
    return str(x)

def clean_text(text: str) -> str:
    # sécurités: on force bien du str
    text = safe_to_str(text)

    text = BeautifulSoup(text, "html.parser").get_text()
    text = text.lower()
    text = re.sub(r"[^a-zàâçéèêëîïôûùüÿñæœ0-9 ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# --------------------------------------------------
# ✅ HEALTH CHECK
# --------------------------------------------------
@app.get("/health")
def health():
    return {"status": "ok"}

# --------------------------------------------------
# ✅ SINGLE PREDICTION
# --------------------------------------------------
@app.post("/predict")
def predict(product: Product):
    designation = safe_to_str(product.designation)
    description = safe_to_str(product.description)

    text = clean_text(f"{designation} {description}").strip()
    if not text:
        raise HTTPException(status_code=400, detail="Text is empty after preprocessing")

    try:
        prediction = model.predict([text])[0]
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

    # prediction peut être numpy/int/str => on sécurise
    try:
        return {"prediction": int(prediction)}
    except Exception:
        return {"prediction": str(prediction)}

# --------------------------------------------------
# ✅ BATCH PREDICTION
# --------------------------------------------------
@app.post("/predict_batch")
def predict_batch(products: List[Product]):
    if len(products) == 0:
        raise HTTPException(status_code=400, detail="Empty input list")

    texts = []
    for p in products:
        designation = safe_to_str(p.designation)
        description = safe_to_str(p.description)
        texts.append(clean_text(f"{designation} {description}"))

    try:
        predictions = model.predict(texts)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

    # predictions peut être numpy array
    try:
        return {"predictions": [int(x) for x in predictions.tolist()]}
    except Exception:
        return {"predictions": predictions.tolist()}
